# Streaming Covariance Protocol Characterization Across `N20x[20,30,40]`

Runs canonical GPU Markov dynamics for three `Nx=20` domain-wall systems and characterizes each cycle online with `cycle_observer`, without saving raw covariance histories.

The notebook requests `10` samples and `50` cycles for both `perfect_correction` and `postselect`. The canonical GPU driver still forces `postselect=True` to one deterministic sample, and that effective sample count is recorded in the saved manifests.

Saved products are derived-only: per-sample-cycle metrics, Frobenius successive differences, averaged strip-entropy curves, ensemble summaries, and a multi-page PDF report.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)

BUNDLE_NAME = 'colab_small_system_testing'

def is_bundle_root(path: Path) -> bool:
    return (path / 'src' / 'classA_U1FGTN_gpu.py').exists() and (path / 'src' / 'streaming_covariance_protocol_characterization_gpu.py').exists()

candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base, base / BUNDLE_NAME])
if IN_COLAB:
    candidates.extend([
        Path('/content/drive/MyDrive') / BUNDLE_NAME,
        Path('/content/drive/MyDrive/class_A_fermionic_adaptive_circuit') / BUNDLE_NAME,
    ])

seen = set()
BUNDLE_ROOT = None
for candidate in candidates:
    candidate = candidate.resolve()
    if candidate in seen:
        continue
    seen.add(candidate)
    if is_bundle_root(candidate):
        BUNDLE_ROOT = candidate
        break
if BUNDLE_ROOT is None:
    raise FileNotFoundError(f'Could not locate {BUNDLE_NAME} with required src files')

SRC_DIR = BUNDLE_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'[setup] bundle root: {BUNDLE_ROOT}')

## 2. Imports

In [ ]:
import gc
import math
import time
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from tqdm.auto import tqdm
import torch

from classA_U1FGTN_gpu import classA_U1FGTN_gpu
from streaming_covariance_protocol_characterization_gpu import (
    HELPER_VERSION,
    PROTOCOL_ORDER,
    StreamingCovarianceCharacterizer,
    build_ensemble_summaries,
    build_entropy_curve_payload,
    case_output_dir,
    config_id,
    create_overview_figure,
    domain_wall_metadata,
    expected_samples_for_config,
    geometry_key,
    rel_to_root,
    run_internal_checks,
    save_dataframe_atomic_csv,
    save_npz_atomic,
    write_json_atomic,
    write_pdf_report,
)

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this Colab GPU campaign.')

torch.set_grad_enabled(False)
plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 180,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

total_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'[setup] helper={HELPER_VERSION}')
print(f'[setup] torch={torch.__version__}')
print(f'[setup] cuda device: {torch.cuda.get_device_name(0)}')
print(f'[setup] total gpu memory: {total_gib:.2f} GiB')
if total_gib < 35.0:
    print('[warn] This notebook is sized for an A100 40GB runtime.')

run_internal_checks()
print('[setup] helper internal checks passed')

## 3. Campaign Configuration

In [ ]:
NX = 20
NY_VALUES = [20, 30, 40]
NSHELL = 1
INIT_MODE = 'default'
REQUESTED_SAMPLES = 10
CYCLES = 50
EARLY_CYCLE_MAX = 20
SNAPSHOT_CYCLES = (20, 21, 30, 40, 50)
FIT_AY_MIN = 8
ALPHA_1 = 1.0
ALPHA_2 = 30.0
DW_TRUNCATION = True
DTYPE = 'complex128'
DEVICE = 'cuda:0'
BACKEND = 'local'
SEQUENCE = 'raster_y'
BATCH_SIZE = 1
TRIAL_ORBITALS = 'X'
N_A = 0.5
AUTOTUNE_ENTROPY = False
ENTROPY_AUTOTUNE_REPEAT = 1
SAMPLE_CHUNK_CANDIDATES = (1, 2, 4, 8, 16)
Y0_CHUNK_CANDIDATES = (1, 2, 4, 6, 8, 10, 12, 16, 20, 'ny')
MEMORY_SAFETY_FRACTION = 0.85
EIGH_MEMORY_MULTIPLIER = 8.0
OVERWRITE = True
WRITE_PDF = True
DISPLAY_INLINE_FIGURES = True

PROTOCOL_CONFIGS = [
    {
        'protocol': 'perfect_correction',
        'postselect': False,
        'perfect_correction': True,
        'samples': REQUESTED_SAMPLES,
    },
    {
        'protocol': 'postselect',
        'postselect': True,
        'perfect_correction': False,
        'samples': REQUESTED_SAMPLES,
    },
]

GEOMETRIES = [{'Nx': NX, 'Ny': ny} for ny in NY_VALUES]
CONFIGS = [
    {
        **geometry,
        'nshell': NSHELL,
        'init_mode': INIT_MODE,
        **protocol_cfg,
    }
    for geometry in GEOMETRIES
    for protocol_cfg in PROTOCOL_CONFIGS
]

GPU_DATA_ROOT = BUNDLE_ROOT / 'gpu_data'
CAMPAIGN_NAME = 'streaming_covariance_protocol_characterization'
OUTPUT_ROOT = GPU_DATA_ROOT / CAMPAIGN_NAME
CAMPAIGN_ID = f'N{NX}_multi_geometry_C{CYCLES}_dwtrunc1_init-{INIT_MODE}_nsh{NSHELL}_S{REQUESTED_SAMPLES}'
CAMPAIGN_ROOT = OUTPUT_ROOT / 'campaigns' / CAMPAIGN_ID
RUNS_ROOT = CAMPAIGN_ROOT / 'runs'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

CANONICAL_ENTRY_POINT = 'classA_U1FGTN_gpu.run_markov_circuit'
SAVED_OBSERVABLES = [
    'frob_successive_delta',
    'entropy_slope fit from y0-averaged strip entropies',
    'real_space_chern',
    'normalized_trace',
    'entropy_curves',
]

display(Markdown('### Campaign config'))
display(pd.DataFrame([{
    'Nx': NX,
    'Ny_values': str(NY_VALUES),
    'nshell': NSHELL,
    'cycles': CYCLES,
    'requested_samples': REQUESTED_SAMPLES,
    'early_cycles': f'1..{EARLY_CYCLE_MAX}',
    'late_cycles': f'{EARLY_CYCLE_MAX + 1}..{CYCLES}',
    'snapshot_cycles': str(SNAPSHOT_CYCLES),
    'fit_Ay_min': FIT_AY_MIN,
    'dw_truncation': DW_TRUNCATION,
    'dtype': DTYPE,
    'sequence': SEQUENCE,
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
    'campaign_root': str(CAMPAIGN_ROOT),
}]))
print(json.dumps({
    'campaign_name': CAMPAIGN_NAME,
    'campaign_id': CAMPAIGN_ID,
    'configs': CONFIGS,
    'saved_cycles': list(range(1, CYCLES + 1)),
    'covariance_history_saved': False,
    'protocol_note': 'postselect=True requests 10 samples but the canonical GPU driver forces samples=1.',
}, indent=2))

## 4. Shared Notebook Helpers

In [ ]:
def update_gpu_data_index() -> None:
    index_path = GPU_DATA_ROOT / 'index.json'
    if index_path.exists():
        payload = json.loads(index_path.read_text())
    else:
        payload = {'bundle_root': str(BUNDLE_ROOT), 'gpu_data_root': str(GPU_DATA_ROOT), 'campaigns': []}
    campaigns = payload.setdefault('campaigns', [])
    campaigns[:] = [item for item in campaigns if item.get('name') != CAMPAIGN_NAME]
    campaigns.append({
        'name': CAMPAIGN_NAME,
        'kind': 'streaming_covariance_characterization',
        'manifest': f'{CAMPAIGN_NAME}/latest_campaign.json',
    })
    payload['bundle_root'] = str(BUNDLE_ROOT)
    payload['gpu_data_root'] = str(GPU_DATA_ROOT)
    write_json_atomic(index_path, payload)

def assert_overwrite_allowed(path: Path) -> None:
    if path.exists() and not OVERWRITE:
        raise FileExistsError(f'Path already exists and OVERWRITE=False: {path}')

def run_config_payload(cfg: dict[str, Any], *, actual_samples: int, dw_meta: dict[str, Any]) -> dict[str, Any]:
    return {
        'Nx': int(cfg['Nx']),
        'Ny': int(cfg['Ny']),
        'nshell': int(cfg['nshell']),
        'init_mode': str(cfg['init_mode']),
        'protocol': str(cfg['protocol']),
        'postselect': bool(cfg['postselect']),
        'perfect_correction': bool(cfg['perfect_correction']),
        'samples_requested': int(cfg['samples']),
        'samples_actual': int(actual_samples),
        'cycles': int(CYCLES),
        'fit_ay_min': int(FIT_AY_MIN),
        'early_cycle_max': int(EARLY_CYCLE_MAX),
        'snapshot_cycles': list(SNAPSHOT_CYCLES),
        'alpha_1': float(ALPHA_1),
        'alpha_2': float(ALPHA_2),
        'dw_truncation': bool(DW_TRUNCATION),
        'dtype': DTYPE,
        'device': DEVICE,
        'backend': BACKEND,
        'sequence': SEQUENCE,
        'batch_size_requested': BATCH_SIZE,
        'n_a': float(N_A),
        'trial_orbitals': TRIAL_ORBITALS,
        'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
        'covariance_history_saved': False,
        'observable_collection_mode': 'cycle_observer_streaming',
        'saved_observables': SAVED_OBSERVABLES,
        'dw_loc': dw_meta['dw_loc'],
        'topological_x_range': dw_meta['topological_x_range'],
        'trivial_x_segments': dw_meta['trivial_x_segments'],
    }

def maybe_display_fig(fig, title: str) -> None:
    if not DISPLAY_INLINE_FIGURES:
        plt.close(fig)
        return
    display(Markdown(f'### {title}'))
    display(fig)
    plt.close(fig)

## 5. Run Streaming Characterization Campaign

In [ ]:
def run_one_config(cfg: dict[str, Any]) -> dict[str, Any]:
    out_dir = case_output_dir(RUNS_ROOT, cfg)
    out_dir.mkdir(parents=True, exist_ok=True)
    expected_samples = expected_samples_for_config(cfg)
    protocol = str(cfg['protocol'])
    print(
        f"[config] Nx={cfg['Nx']} Ny={cfg['Ny']} nshell={cfg['nshell']} init={cfg['init_mode']} "
        f"protocol={protocol} requested_samples={cfg['samples']} cycles={CYCLES} output={out_dir}"
    )

    model = classA_U1FGTN_gpu(
        Nx=int(cfg['Nx']),
        Ny=int(cfg['Ny']),
        DW=True,
        nshell=int(cfg['nshell']),
        filling_frac=0.5,
        alpha_1=ALPHA_1,
        alpha_2=ALPHA_2,
        trial_orbitals=TRIAL_ORBITALS,
        dw_truncation=DW_TRUNCATION,
        triv_region_local_mode=False,
        device=DEVICE,
        dtype=DTYPE,
        backend=BACKEND,
    )

    dw_meta = domain_wall_metadata(model)
    run_payload_stub = run_config_payload(cfg, actual_samples=expected_samples, dw_meta=dw_meta)
    observer = StreamingCovarianceCharacterizer(
        nx=int(cfg['Nx']),
        ny=int(cfg['Ny']),
        cycles=CYCLES,
        samples_expected=expected_samples,
        protocol=protocol,
        nshell=int(cfg['nshell']),
        fit_ay_min=FIT_AY_MIN,
        early_cycle_max=EARLY_CYCLE_MAX,
        snapshot_cycles=SNAPSHOT_CYCLES,
        dw_loc=dw_meta['dw_loc'],
        autotune_entropy=AUTOTUNE_ENTROPY,
        sample_chunk_candidates=SAMPLE_CHUNK_CANDIDATES,
        y0_chunk_candidates=Y0_CHUNK_CANDIDATES,
        autotune_repeat=ENTROPY_AUTOTUNE_REPEAT,
        memory_safety_fraction=MEMORY_SAFETY_FRACTION,
        eigh_memory_multiplier=EIGH_MEMORY_MULTIPLIER,
    )

    observer_pbar = tqdm(
        total=expected_samples * CYCLES,
        desc=f"observer {config_id(cfg['Nx'], cfg['Ny'], protocol)}",
        unit='sample-cycle',
        leave=False,
    )
    cycle_observer = observer.make_cycle_observer(progress_bar=observer_pbar)
    try:
        result = model.run_markov_circuit(
            G_history=False,
            progress=False,
            cycles=CYCLES,
            postselect=bool(cfg['postselect']),
            perfect_correction=bool(cfg['perfect_correction']),
            samples=int(cfg['samples']),
            init_mode=str(cfg['init_mode']),
            save=False,
            save_init=False,
            save_history_stride=None,
            n_a=N_A,
            sequence=SEQUENCE,
            batch_size=BATCH_SIZE,
            return_data=False,
            cycle_observer=cycle_observer,
        )
    finally:
        observer_pbar.close()

    actual_samples = int(result['samples'])
    if actual_samples != expected_samples:
        raise RuntimeError(f'Expected {expected_samples} samples but run returned {actual_samples}.')
    observer.finalize(actual_samples=actual_samples)

    metrics_df = observer.metrics_dataframe().sort_values(['sample_index', 'cycle_label']).reset_index(drop=True)
    frob_df = observer.frob_dataframe().sort_values(['sample_index', 'cycle_label']).reset_index(drop=True)
    run_payload = run_config_payload(cfg, actual_samples=actual_samples, dw_meta=dw_meta)

    metrics_path = out_dir / 'per_sample_cycle_metrics.csv'
    frob_path = out_dir / 'frob_successive_deltas.csv'
    entropy_path = out_dir / 'entropy_curves.npz'
    summary_path = out_dir / 'run_summary.json'
    latest_path = out_dir / 'latest_run.json'
    for path in [metrics_path, frob_path, entropy_path, summary_path, latest_path]:
        assert_overwrite_allowed(path)
    save_dataframe_atomic_csv(metrics_df, metrics_path)
    save_dataframe_atomic_csv(frob_df, frob_path)
    save_npz_atomic(entropy_path, **observer.entropy_curve_npz_payload(config=run_payload))

    run_summary = {
        **run_payload,
        'run_dir': str(out_dir),
        'run_dir_relative': rel_to_root(out_dir, GPU_DATA_ROOT),
        'per_sample_cycle_metrics_relative': rel_to_root(metrics_path, GPU_DATA_ROOT),
        'frob_successive_deltas_relative': rel_to_root(frob_path, GPU_DATA_ROOT),
        'entropy_curves_relative': rel_to_root(entropy_path, GPU_DATA_ROOT),
        'batch_size': int(result['batch_size']),
        'batch_size_mode': result['batch_size_mode'],
        'batch_size_auto_info': result.get('batch_size_auto_info'),
        'metrics_row_count': int(len(metrics_df)),
        'frob_row_count': int(len(frob_df)),
        'helper_version': HELPER_VERSION,
        **observer.run_summary_metrics(),
    }
    write_json_atomic(summary_path, run_summary)
    write_json_atomic(
        latest_path,
        {
            'run_summary_path': str(summary_path),
            'run_summary_path_relative': rel_to_root(summary_path, GPU_DATA_ROOT),
            **run_summary,
        },
    )

    result_row = {
        'config_id': config_id(cfg['Nx'], cfg['Ny'], protocol),
        'geometry_key': geometry_key(cfg['Nx'], cfg['Ny']),
        'protocol': protocol,
        'Nx': int(cfg['Nx']),
        'Ny': int(cfg['Ny']),
        'nshell': int(cfg['nshell']),
        'samples_requested': int(cfg['samples']),
        'samples_actual': int(actual_samples),
        'batch_size': int(result['batch_size']),
        'batch_size_mode': result['batch_size_mode'],
        'run_dir_relative': rel_to_root(out_dir, GPU_DATA_ROOT),
        'run_summary_relative': rel_to_root(summary_path, GPU_DATA_ROOT),
        'dw_loc': dw_meta['dw_loc'],
        'topological_x_range': dw_meta['topological_x_range'],
        'trivial_x_segments': dw_meta['trivial_x_segments'],
        'metrics_df': metrics_df,
        'frob_df': frob_df,
        'entropy_curves': observer.entropy_curves[:actual_samples].copy(),
        'ay_values': observer.ay_values.copy(),
        'cycles': int(CYCLES),
        'actual_samples_for_curves': int(actual_samples),
    }

    del model, observer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result_row

all_results = []
for cfg in tqdm(CONFIGS, desc='streaming configs', unit='config'):
    all_results.append(run_one_config(cfg))

metrics_df = pd.concat([row['metrics_df'] for row in all_results], ignore_index=True).sort_values(
    ['Nx', 'Ny', 'protocol', 'sample_index', 'cycle_label']
).reset_index(drop=True)
frob_df = pd.concat([row['frob_df'] for row in all_results], ignore_index=True).sort_values(
    ['Nx', 'Ny', 'protocol', 'sample_index', 'cycle_label']
).reset_index(drop=True)
ensemble_df = build_ensemble_summaries(
    metrics_df,
    frob_df,
    early_cycle_max=EARLY_CYCLE_MAX,
    cycles=CYCLES,
    snapshot_cycles=SNAPSHOT_CYCLES,
)
curve_payload = build_entropy_curve_payload(
    metrics_df,
    [
        {
            'config_id': row['config_id'],
            'entropy_curves': row['entropy_curves'],
            'ay_values': row['ay_values'],
            'actual_samples': row['actual_samples_for_curves'],
            'cycles': row['cycles'],
        }
        for row in all_results
    ],
)

metrics_csv = CAMPAIGN_ROOT / 'per_sample_cycle_metrics.csv'
frob_csv = CAMPAIGN_ROOT / 'frob_successive_deltas.csv'
summary_csv = CAMPAIGN_ROOT / 'ensemble_summaries.csv'
curves_npz = CAMPAIGN_ROOT / 'entropy_curves.npz'
for path in [metrics_csv, frob_csv, summary_csv, curves_npz]:
    assert_overwrite_allowed(path)
save_dataframe_atomic_csv(metrics_df, metrics_csv)
save_dataframe_atomic_csv(frob_df, frob_csv)
save_dataframe_atomic_csv(ensemble_df, summary_csv)
save_npz_atomic(curves_npz, **curve_payload)

display(Markdown('### Campaign run summary'))
display(
    pd.DataFrame([
        {
            'Nx': row['Nx'],
            'Ny': row['Ny'],
            'protocol': row['protocol'],
            'samples_requested': row['samples_requested'],
            'samples_actual': row['samples_actual'],
            'batch_size': row['batch_size'],
            'batch_size_mode': row['batch_size_mode'],
            'dw_loc': row['dw_loc'],
            'run_dir_relative': row['run_dir_relative'],
        }
        for row in all_results
    ]).sort_values(['Nx', 'Ny', 'protocol']).reset_index(drop=True)
)

## 6. Figures And Campaign Manifest

In [ ]:
pdf_path = CAMPAIGN_ROOT / 'covariance_protocol_characterization.pdf'
pdf_pbar = tqdm(total=1, desc='pdf report', unit='report', leave=False)
try:
    if WRITE_PDF:
        assert_overwrite_allowed(pdf_path)
        pdf_pbar.reset(total=15)
        write_pdf_report(
            pdf_path,
            metrics_df,
            frob_df,
            geometry_order=[(NX, ny) for ny in NY_VALUES],
            protocol_order=PROTOCOL_ORDER,
            early_cycle_max=EARLY_CYCLE_MAX,
            cycles=CYCLES,
            snapshot_cycles=SNAPSHOT_CYCLES,
            progress_bar=pdf_pbar,
        )
finally:
    pdf_pbar.close()

campaign_results = [
    {
        'config_id': row['config_id'],
        'geometry_key': row['geometry_key'],
        'protocol': row['protocol'],
        'Nx': row['Nx'],
        'Ny': row['Ny'],
        'nshell': row['nshell'],
        'samples_requested': row['samples_requested'],
        'samples_actual': row['samples_actual'],
        'batch_size': row['batch_size'],
        'batch_size_mode': row['batch_size_mode'],
        'run_dir_relative': row['run_dir_relative'],
        'run_summary_relative': row['run_summary_relative'],
        'dw_loc': row['dw_loc'],
        'topological_x_range': row['topological_x_range'],
        'trivial_x_segments': row['trivial_x_segments'],
    }
    for row in all_results
]

campaign_manifest = {
    'campaign_name': CAMPAIGN_NAME,
    'campaign_id': CAMPAIGN_ID,
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
    'helper_version': HELPER_VERSION,
    'bundle_root': str(BUNDLE_ROOT),
    'gpu_data_root': str(GPU_DATA_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'campaign_root': str(CAMPAIGN_ROOT),
    'runs_root': str(RUNS_ROOT),
    'output_root_relative': rel_to_root(OUTPUT_ROOT, GPU_DATA_ROOT),
    'campaign_root_relative': rel_to_root(CAMPAIGN_ROOT, GPU_DATA_ROOT),
    'runs_root_relative': rel_to_root(RUNS_ROOT, GPU_DATA_ROOT),
    'geometries': GEOMETRIES,
    'configs': CONFIGS,
    'results': campaign_results,
    'requested_samples': REQUESTED_SAMPLES,
    'cycles': CYCLES,
    'saved_cycles': list(range(1, CYCLES + 1)),
    'fit_ay_min': FIT_AY_MIN,
    'early_cycle_max': EARLY_CYCLE_MAX,
    'late_cycles': list(range(EARLY_CYCLE_MAX + 1, CYCLES + 1)),
    'snapshot_cycles': list(SNAPSHOT_CYCLES),
    'dtype': DTYPE,
    'init_mode': INIT_MODE,
    'dw_truncation': DW_TRUNCATION,
    'trial_orbitals': TRIAL_ORBITALS,
    'backend': BACKEND,
    'sequence': SEQUENCE,
    'covariance_history_saved': False,
    'observable_collection_mode': 'cycle_observer_streaming',
    'saved_observables': SAVED_OBSERVABLES,
    'campaign_outputs': {
        'per_sample_cycle_metrics_relative': rel_to_root(metrics_csv, GPU_DATA_ROOT),
        'frob_successive_deltas_relative': rel_to_root(frob_csv, GPU_DATA_ROOT),
        'ensemble_summaries_relative': rel_to_root(summary_csv, GPU_DATA_ROOT),
        'entropy_curves_relative': rel_to_root(curves_npz, GPU_DATA_ROOT),
        'pdf_report_relative': rel_to_root(pdf_path, GPU_DATA_ROOT) if WRITE_PDF else None,
    },
    'note': 'postselect=True requests 10 samples but the canonical GPU driver forces samples=1.',
}
campaign_manifest_path = CAMPAIGN_ROOT / 'campaign_manifest.json'
write_json_atomic(campaign_manifest_path, campaign_manifest)

latest_campaign_path = OUTPUT_ROOT / 'latest_campaign.json'
write_json_atomic(
    latest_campaign_path,
    {
        'campaign_name': CAMPAIGN_NAME,
        'campaign_id': CAMPAIGN_ID,
        'manifest': rel_to_root(campaign_manifest_path, GPU_DATA_ROOT),
        'campaign_manifest_path': str(campaign_manifest_path),
        'campaign_manifest_path_relative': rel_to_root(campaign_manifest_path, GPU_DATA_ROOT),
    },
)
update_gpu_data_index()
print(f'[done] campaign manifest: {campaign_manifest_path}')
print(f'[done] latest campaign pointer: {latest_campaign_path}')
if WRITE_PDF:
    print(f'[done] pdf report: {pdf_path}')

if DISPLAY_INLINE_FIGURES:
    for ny in NY_VALUES:
        fig = create_overview_figure(
            geometry_key(NX, ny),
            metrics_df,
            frob_df,
            protocol_order=PROTOCOL_ORDER,
            early_cycle_max=EARLY_CYCLE_MAX,
            cycles=CYCLES,
        )
        maybe_display_fig(fig, f'Overview: N{NX}x{ny}')

## 7. Disconnect Runtime

In [ ]:
if IN_COLAB:
    from google.colab import runtime  # type: ignore
    runtime.unassign()
else:
    print('[done] not running in Colab; runtime disconnect skipped')